# RL Activation Steering — Boundary Responses Only
## TriviaQA — Everything from scratch

---

## The idea

When a model hallucinates, its hidden state at the critical layer L* drifts toward a hallucination region in activation space. This notebook trains an RL policy to steer **only the ambiguous (boundary) responses** back toward the factual region.

```
Confident factual  (probe < 0.45) → leave alone
Boundary zone      (probe 0.45–0.65) → STEER with Δh
Confident hallu    (probe > 0.65) → leave alone (too far gone)
```

---

## Three improvements over previous version

| # | What | Why |
|---|---|---|
| 1 | **Boundary-only steering** | Small push can flip ambiguous responses. Confident ones don't benefit. |
| 2 | **Probe direction reward** | `probe.coef_[0]` generalizes better than a dataset-specific centroid |
| 3 | **L2 regularizer** | `loss = -reward + λ‖Δh‖²` prevents large perturbations that corrupt representations |


## Before running

1. Replace `your_hf_token_here` in **Cell 1**
2. Runtime → Change runtime type → **A100 GPU**
3. Runtime → **Run all**

> All results saved to `/content/drive/MyDrive/facr_triviaqa/`

### CELL 0 — Install dependencies

Run this first. Restart runtime if prompted.

In [ ]:
# ── CELL 0 — Install dependencies ────────────────────────────────
# Run this first. Restart runtime if prompted, then run all again.

!pip install transformers datasets accelerate tqdm scikit-learn matplotlib -q


### CELL 1 — Imports + frozen Llama-3.2-3B loading

Replace `your_hf_token_here` with your HuggingFace token before running.

In [ ]:
# ── CELL 1 — Imports and model loading ───────────────────────────
#
# Loads all libraries and frozen Llama-3.2-3B-Instruct.
# output_hidden_states=True lets us extract activations
# at any of the 28 transformer layers.
#
# REPLACE "your_hf_token_here" with your HuggingFace token.
# Get one at: https://huggingface.co/settings/tokens

import os, json, re, string, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # already mounted

SAVE_DIR = '/content/drive/MyDrive/facr_triviaqa'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

from huggingface_hub import login
login("<REDACTED>")  # <── REPLACE THIS

MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE      = "cuda"
NUM_LAYERS  = 28
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"\nDevice : {DEVICE} | GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)
model.eval()
print(f"Model  : {MODEL_ID} ✓ (frozen — never modified)")
print(f"Layers : {NUM_LAYERS + 1} "
      f"(0=embedding, 1-{NUM_LAYERS}=transformer blocks)")


### CELL 2 — Generate TriviaQA data from scratch (~36 min)

Generates 2000 TriviaQA responses and labels with token F1. Always from scratch. Checkpoints every 100 questions in case of disconnect.

In [ ]:
# ── CELL 2 — Generate TriviaQA data ──────────────────────────────
#
# Generates responses for 500 TriviaQA questions, 4 per question
# = 2000 total samples. Labels each with token F1 against gold.
#
# factuality_score > 0.5 → label = 1 (FACTUAL)
# factuality_score ≤ 0.5 → label = 0 (HALLUCINATED)
#
# Always regenerates from scratch. Checkpoints every 100 questions.
# Expected runtime: ~36 minutes on A100.

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b',
                  lambda m: m.group(0).rstrip('s'), text)
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    p_tok = normalize(pred).split()
    r_tok = normalize(ref).split()
    if not p_tok or not r_tok: return 0.0
    common = set(p_tok) & set(r_tok)
    if not common: return 0.0
    p = len(common) / len(p_tok)
    r = len(common) / len(r_tok)
    return 2 * p * r / (p + r)

def generate_response(question):
    prompt = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=200
    ).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=40,
            temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(
        new_tokens, skip_special_tokens=True).strip()
    return decoded.split('\n')[0].split('.')[0].strip()

print("Generating TriviaQA responses from scratch (~36 min)...\n")

dataset      = load_dataset(
    "trivia_qa", "rc.nocontext", split="train[:500]")
labeled_data = []
dist         = {'factual': 0, 'hallu': 0}

for i, example in enumerate(tqdm(dataset)):
    question   = example['question']
    ans        = example['answer']
    prompt     = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    candidates = []
    if ans.get('value'): candidates.append(ans['value'])
    for a in ans.get('aliases', []):
        if isinstance(a, str): candidates.append(a)
    for a in ans.get('normalized_aliases', []):
        if isinstance(a, str): candidates.append(a)

    for _ in range(4):
        resp  = generate_response(question)
        score = max((token_f1(resp, ref)
                     for ref in candidates), default=0.0)
        labeled_data.append({
            'question':         question,
            'response':         resp,
            'factuality_score': float(score),
            'prompt':           prompt
        })
        if score > 0.5: dist['factual'] += 1
        else:           dist['hallu']   += 1

    if (i + 1) % 100 == 0:
        ckpt = f'{SAVE_DIR}/ckpt_{i+1}.json'
        with open(ckpt, 'w') as f:
            json.dump(labeled_data, f)
        print(f"  Checkpoint {i+1}/500 | "
              f"factual={dist['factual']} hallu={dist['hallu']}")

data_path = f'{SAVE_DIR}/labeled_data.json'
with open(data_path, 'w') as f:
    json.dump(labeled_data, f)

total = len(labeled_data)
print(f"\nDone! {total} samples | "
      f"factual={dist['factual']} ({100*dist['factual']/total:.1f}%) | "
      f"hallu={dist['hallu']} ({100*dist['hallu']/total:.1f}%)")

random.seed(RANDOM_SEED)
random.shuffle(labeled_data)
TRAIN_DATA = labeled_data[:1600]
EVAL_DATA  = labeled_data[1600:]
print(f"Train: {len(TRAIN_DATA)} | Eval: {len(EVAL_DATA)}")


### CELL 3 — Find critical layer L* + train linear probe

Sweeps all 28 layers to find L* automatically. No hardcoded layer numbers. Trains a linear probe at L* and stores it in `probe` for use in Cell 6.

In [ ]:
# ── CELL 3 — Find critical layer L* automatically ────────────────
#
# Sweeps all 28 layers and finds L* — the layer where a linear
# probe best separates factual from hallucinated hidden states.
#
# THREE-WAY SPLIT prevents leakage:
#   20% selection → find L*
#   60% training  → train probe at L*
#   20% eval      → measure final AUROC
#
# L* is stored in TARGET_LAYER and reused in all later cells.
# The trained probe is stored in `probe` and used in Cell 6.

def extract_hidden_states(data, layer_idx, cutoff_pct=1.0):
    """
    Extract last-token hidden state at layer_idx.
    cutoff_pct < 1.0 simulates early detection.
    """
    hiddens, labels = [], []
    for item in data:
        full_text = item['prompt'] + " " + item['response']
        inputs    = tokenizer(
            full_text, return_tensors='pt',
            truncation=True, max_length=256
        ).to(DEVICE)
        if cutoff_pct < 1.0:
            total  = inputs['input_ids'].shape[1]
            cutoff = max(4, int(total * cutoff_pct))
            inputs = {k: v[:, :cutoff] for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
        h = out.hidden_states[layer_idx][0, -1, :].float().cpu()
        hiddens.append(h)
        labels.append(1 if item['factuality_score'] > 0.5 else 0)
    return torch.stack(hiddens), torch.tensor(labels)

def quick_auroc(h, l, cv=3):
    X = h.numpy(); y = l.numpy()
    if len(np.unique(y)) < 2: return 0.5
    return float(cross_val_score(
        LogisticRegression(max_iter=300),
        X, y, cv=cv, scoring='roc_auc').mean())

def robust_auroc(h, l, cv=10):
    X   = h.numpy(); y = l.numpy()
    if len(np.unique(y)) < 2: return 0.5, 0.0
    skf = StratifiedKFold(
        n_splits=cv, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X[tr], y[tr])
        scores.append(roc_auc_score(
            y[te], clf.predict_proba(X[te])[:, 1]))
    return float(np.mean(scores)), float(np.std(scores))

print("Finding critical layer L* on TriviaQA...")
sel_data = TRAIN_DATA[:int(0.2 * len(TRAIN_DATA))]

layer_aurocs = {}
for layer_idx in range(NUM_LAYERS + 1):
    h, l = extract_hidden_states(sel_data, layer_idx, 1.0)
    layer_aurocs[layer_idx] = quick_auroc(h, l, cv=3)

TARGET_LAYER = max(layer_aurocs, key=layer_aurocs.get)

print(f"Layer AUROC (every 4 layers):")
for i in range(0, NUM_LAYERS + 1, 4):
    marker = " ← L*" if i == TARGET_LAYER else ""
    print(f"  Layer {i:2d}: {layer_aurocs[i]:.4f}{marker}")
print(f"\n★ L* = {TARGET_LAYER} "
      f"({TARGET_LAYER/NUM_LAYERS*100:.0f}% depth)")

# train probe at L*
probe_train = TRAIN_DATA[
    int(0.2*len(TRAIN_DATA)):int(0.8*len(TRAIN_DATA))]
h_tr, l_tr  = extract_hidden_states(probe_train, TARGET_LAYER, 1.0)
probe        = LogisticRegression(max_iter=1000, C=1.0)
probe.fit(h_tr.numpy(), l_tr.numpy())

h_ev, l_ev   = extract_hidden_states(EVAL_DATA, TARGET_LAYER, 1.0)
proba         = probe.predict_proba(h_ev.numpy())[:, 1]
full_auroc    = roc_auc_score(l_ev.numpy(), proba)
_, full_std   = robust_auroc(h_ev, l_ev, cv=10)

print(f"\nProbe AUROC at L*={TARGET_LAYER}: "
      f"{full_auroc:.4f} ± {full_std:.4f}")
print(f"Probe trained and stored in `probe` ✓")

with open(f'{SAVE_DIR}/layer_sweep.json', 'w') as f:
    json.dump({
        'target_layer': TARGET_LAYER,
        'layer_aurocs': layer_aurocs,
        'full_auroc':   full_auroc,
        'full_std':     full_std,
    }, f, indent=2)


### CELL 4 — Early detection sweep (10% to 100% tokens)

Tests detection at 10%, 20%, 30%, 50%, 100% of tokens. Answers: how early can we detect a hallucination?

In [ ]:
# ── CELL 4 — Early detection sweep ───────────────────────────────
#
# Tests detection at 10%, 20%, 30%, 50%, 100% of tokens.
# Answers: how early in generation can we detect hallucination?

CUTOFFS = [0.10, 0.20, 0.30, 0.50, 1.00]

print("Early detection sweep...")
print(f"{'Cutoff':>8} {'AUROC':>8} {'±Std':>8} {'Signal':>12}")
print("-" * 44)

early_results = {}
for cutoff in CUTOFFS:
    h_c, l_c  = extract_hidden_states(
        EVAL_DATA, TARGET_LAYER, cutoff)
    mean, std = robust_auroc(h_c, l_c, cv=10)
    early_results[cutoff] = {'auroc': mean, 'std': std}
    sig = ("strong ✓" if mean > 0.70 else
           "reliable ~" if mean > 0.60 else
           "weak" if mean > 0.55 else "chance ✗")
    print(f"{int(cutoff*100):>7}%   {mean:.4f}   ±{std:.4f}  {sig:>12}")

earliest = next((c for c in sorted(CUTOFFS)
                 if early_results[c]['auroc'] > 0.60), None)
print(f"\nEarliest reliable detection: "
      f"{int(earliest*100) if earliest else 'none'}% tokens")

with open(f'{SAVE_DIR}/early_detection.json', 'w') as f:
    json.dump(early_results, f, indent=2)


### CELL 5 — RL policy + boundary zone + probe direction definitions

**Three key design choices:**

1. **Boundary zone [0.45, 0.65]** — Only steer samples near the decision boundary. Confident hallucinations (>0.65) and confident factuals (<0.45) are left alone. Only ambiguous responses in the zone get steered.

2. **Probe direction reward** — Reward = alignment of Δh with `probe.coef_[0]`. This direction maximally separates factual from hallucinated and generalizes better than a dataset-specific centroid.

3. **L2 regularizer** — `loss = -reward + λ * ||Δh||²` penalizes large perturbations on ALL samples, preventing the policy from corrupting the representation.

In [ ]:
# ── CELL 5 — RL steering policy definitions ──────────────────────
#
# THREE KEY DESIGN CHOICES:
#
# 1. BOUNDARY ZONE [0.45, 0.65]
#    Only steer samples where probe score is in this range.
#    These are the ambiguous responses near the decision boundary.
#    A small push can flip them to the factual side.
#    Confident responses (score < 0.45 or > 0.65) are left alone.
#
# 2. PROBE DIRECTION REWARD
#    Reward = alignment of Δh with probe.coef_[0].
#    probe.coef_[0] is the direction that MAXIMALLY separates
#    factual from hallucinated — better than centroid because:
#      - Centroid is dataset-specific (doesn't generalize)
#      - Probe direction is learned to maximize separation
#      - Same direction works on train AND eval data
#
# 3. L2 REGULARIZER
#    loss = -reward + λ * ||Δh||²
#    Penalizes large steering vectors on ALL samples.
#    Prevents wild perturbations that corrupt the representation.
#    Encourages the policy to find the MINIMAL correction.

BOUNDARY_LOW  = 0.45   # below this = confident factual → skip
BOUNDARY_HIGH = 0.65   # above this = confident hallu   → skip
LAMBDA_L2     = 0.1    # L2 regularization strength


class SteeringPolicy(nn.Module):
    """
    MLP that maps hidden state h → steering vector Δh.

    Input  : h ∈ R^hidden_dim  (hidden state at L*)
    Output : Δh ∈ R^hidden_dim (small perturbation)

    Steered state: h_new = h + Δh

    scale parameter limits maximum perturbation magnitude.
    tanh keeps direction bounded regardless of scale.
    """
    def __init__(self, hidden_dim=3072, scale=0.05):
        super().__init__()
        self.scale = scale
        self.net   = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, hidden_dim)
        )

    def forward(self, h):
        """Compute Δh — bounded by scale."""
        delta = self.net(h)
        return torch.tanh(delta) * self.scale

    def steer(self, h):
        """Return h + Δh."""
        return h + self.forward(h)


def get_probe_direction(probe, device='cuda'):
    """
    Extract probe's learned separation direction.
    This vector points from hallucinated toward factual region.
    Using this as the steering target is better than centroid
    because it was optimized for maximum class separation.
    """
    direction = torch.tensor(
        probe.coef_[0], dtype=torch.float32
    ).to(device)
    return F.normalize(direction, dim=-1)


print("RL boundary steering framework loaded ✓")
print(f"\nBoundary zone     : probe ∈ [{BOUNDARY_LOW}, {BOUNDARY_HIGH}]")
print(f"Reward signal     : alignment with probe direction")
print(f"L2 regularizer    : λ = {LAMBDA_L2}")
print(f"Scale             : 0.05 (small — boundary needs gentle push)")
print(f"\nWhy boundary only?")
print(f"  Confident hallu (>{BOUNDARY_HIGH}) → too far, skip")
print(f"  Confident fact  (<{BOUNDARY_LOW}) → correct, skip")
print(f"  Boundary zone   → small push can flip to factual ✓")


### CELL 6 — Train boundary steering with L2 regularizer (~15 min)

**Training loop:**
- For boundary samples (probe ∈ [0.45, 0.65]): reward = cosine alignment of Δh with probe direction, weighted by distance to boundary center
- For all samples: L2 penalty on ||Δh||²
- Watch `boundary_gap` → should increase, `|Δh|` → should stay small

In [ ]:
import numpy as np

# ── CELL 6 — Train boundary steering with L2 regularizer ─────────
#
# TRAINING LOOP:
#   For each boundary sample (probe score ∈ [0.45, 0.65]):
#     1. Compute Δh via policy
#     2. Reward = cosine alignment of Δh with probe direction
#     3. L2 penalty = λ * ||Δh||²  (regularizer)
#     4. Loss = -reward + L2_penalty
#     5. Boundary weight = highest at probe=0.55, lower at edges
#
#   For non-boundary samples:
#     Only apply L2 penalty (don't steer, but prevent large Δh)
#
# WHAT TO WATCH:
#   boundary_gap → projection gap for boundary samples only
#                  should INCREASE (boundary samples moving to factual side)
#   steer_rew    → should be POSITIVE (correct direction)
#   AUROC        → on training data for reference
#
# Expected runtime: ~15 minutes

print("Pre-extracting training hidden states...")
h_train_all, l_train_all = extract_hidden_states(
    TRAIN_DATA, TARGET_LAYER, 1.0)
h_train_dev  = h_train_all.to(DEVICE)
probe_scores = probe.predict_proba(
    h_train_all.numpy())[:, 1]

# get probe direction
probe_dir = get_probe_direction(probe, DEVICE)

# verify probe direction points factual → hallu
proj     = h_train_all @ probe_dir.cpu()
proj_f   = proj[l_train_all == 1].mean().item()
proj_h   = proj[l_train_all == 0].mean().item()
if proj_f < proj_h:
    probe_dir = -probe_dir   # flip if needed
    proj_f, proj_h = proj_h, proj_f
    print("Probe direction flipped ✓")

print(f"Probe direction check:")
print(f"  Factual projection  : {proj_f:+.4f}")
print(f"  Hallu projection    : {proj_h:+.4f}")
print(f"  Gap                 : {proj_f-proj_h:+.4f} ✓")

# identify boundary samples
boundary_mask = ((probe_scores >= BOUNDARY_LOW) &
                 (probe_scores <= BOUNDARY_HIGH))
n_boundary    = boundary_mask.sum()
print(f"\nTraining boundary samples: {n_boundary}/{len(probe_scores)} "
      f"({100*n_boundary/len(probe_scores):.1f}%)")

# initial boundary gap
h_b      = h_train_all[boundary_mask]
l_b      = l_train_all[boundary_mask]
proj_b   = h_b @ probe_dir.cpu()
init_pf  = proj_b[l_b==1].mean().item() if (l_b==1).sum()>0 else 0
init_ph  = proj_b[l_b==0].mean().item() if (l_b==0).sum()>0 else 0
init_gap = init_pf - init_ph
print(f"Initial boundary gap: {init_gap:+.4f} "
      f"(should INCREASE during training)")

# initialize policy
hidden_dim = h_train_all.shape[1]
policy     = SteeringPolicy(hidden_dim=hidden_dim, scale=0.05).to(DEVICE)
optimizer  = torch.optim.Adam(policy.parameters(), lr=5e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30)

total_params = sum(p.numel() for p in policy.parameters())
print(f"\nPolicy params  : {total_params:,}")
print(f"Scale          : 0.05")
print(f"Learning rate  : 5e-4")
print(f"L2 lambda      : {LAMBDA_L2}")
print(f"Boundary zone  : [{BOUNDARY_LOW}, {BOUNDARY_HIGH}]")
print(f"\nTraining...\n")

best_gap   = init_gap
best_epoch = 0
train_log  = []

for epoch in range(30):
    policy.train()
    steer_rews = []
    n_steered  = 0

    idx = torch.randperm(len(h_train_dev))

    for i in idx:
        h           = h_train_dev[i].unsqueeze(0)
        probe_score = float(probe_scores[i])
        delta       = policy(h)              # compute Δh
        h_new       = h + delta              # steered state

        # L2 regularizer — applied to ALL samples
        # penalizes large perturbations regardless of label
        l2_penalty = LAMBDA_L2 * (delta ** 2).mean()

        if BOUNDARY_LOW <= probe_score <= BOUNDARY_HIGH:
            # BOUNDARY SAMPLE — steer toward factual side
            # reward = cosine alignment of Δh with probe direction
            reward_t = F.cosine_similarity(
                F.normalize(delta + 1e-8, dim=-1),
                probe_dir.unsqueeze(0)
            )

            # boundary weight: highest at center (0.55), lower at edges
            # this focuses training on the most ambiguous samples
            bw   = 1.0 - abs(probe_score - 0.55) / 0.10
            loss = -(reward_t * bw).mean() + l2_penalty

            steer_rews.append(reward_t.item())
            n_steered += 1
        else:
            # NON-BOUNDARY SAMPLE — only apply L2 regularizer
            # this teaches the policy to output small Δh
            # for samples it shouldn't be steering
            loss = l2_penalty

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()

    scheduler.step()

    # evaluate every 3 epochs
    if (epoch + 1) % 3 == 0:
        policy.eval()
        with torch.no_grad():
            # apply steering to boundary samples only
            h_new_all = h_train_dev.clone()
            for i in range(len(h_train_dev)):
                ps = float(probe_scores[i])
                if BOUNDARY_LOW <= ps <= BOUNDARY_HIGH:
                    h_new_all[i] = policy.steer(
                        h_train_dev[i].unsqueeze(0)).squeeze(0)

            h_new_cpu = h_new_all.cpu()

            # boundary gap after steering
            h_b_new  = h_new_cpu[boundary_mask]
            proj_b_n = h_b_new @ probe_dir.cpu()
            pf_n = proj_b_n[l_b==1].mean().item() \
                   if (l_b==1).sum()>0 else 0
            ph_n = proj_b_n[l_b==0].mean().item() \
                   if (l_b==0).sum()>0 else 0
            new_gap = pf_n - ph_n

            # full AUROC for reference
            clf   = LogisticRegression(max_iter=300)
            clf.fit(h_new_cpu.numpy(), l_train_all.numpy())
            proba = clf.predict_proba(h_new_cpu.numpy())[:, 1]
            auroc = roc_auc_score(l_train_all.numpy(), proba)

            # mean delta magnitude (should be small due to L2)
            all_deltas = []
            for i in range(len(h_train_dev)):
                d = policy(h_train_dev[i].unsqueeze(0))
                all_deltas.append(d.norm().item())
            mean_delta = np.mean(all_deltas)

        policy.train()

        sr = np.mean(steer_rews) if steer_rews else 0
        print(f"  Epoch {epoch+1:3d}/30 | "
              f"boundary_gap={new_gap:+.4f} "
              f"(init={init_gap:+.4f}) | "
              f"AUROC={auroc:.4f} | "
              f"steer_rew={sr:+.3f} | "
              f"|Δh|={mean_delta:.4f} | "
              f"n_steered={n_steered}")

        train_log.append({
            'epoch':        epoch + 1,
            'boundary_gap': new_gap,
            'auroc':        auroc,
            'steer_rew':    sr,
            'mean_delta':   mean_delta,
        })

        if new_gap > best_gap:
            best_gap   = new_gap
            best_epoch = epoch + 1
            torch.save(
                policy.state_dict(),
                f'{SAVE_DIR}/steering_boundary_best.pt')

with open(f'{SAVE_DIR}/training_log_boundary.json', 'w') as f:
    json.dump(train_log, f, indent=2)

print(f"\nTraining complete!")
print(f"Initial boundary gap : {init_gap:+.4f}")
print(f"Best boundary gap    : {best_gap:+.4f} (epoch {best_epoch})")
print(f"Improvement          : {best_gap - init_gap:+.4f}")
print(f"  (positive = boundary samples moved toward factual ✓)")

policy.load_state_dict(
    torch.load(f'{SAVE_DIR}/steering_boundary_best.pt'))
policy.eval()
print(f"Best checkpoint loaded ✓")


### CELL 7 — Evaluate: before vs after — all samples + boundary only

Reports AUROC and projection gap for **all samples** and **boundary samples only**. The boundary-only result is the main finding.

In [ ]:
# ── CELL 7 — Evaluate boundary steering ──────────────────────────
#
# Evaluates on the held-out eval set.
# Reports metrics for:
#   a) ALL samples — overall AUROC
#   b) BOUNDARY SAMPLES ONLY — the main result
#      (these are the samples the policy was designed for)
#
# KEY METRIC: AUROC on boundary samples before vs after steering.
# Even a small improvement here (+0.02) means the policy is
# successfully flipping ambiguous responses to the correct side.

print("Evaluating boundary steering on held-out eval set...")

h_eval_all, l_eval_all = extract_hidden_states(
    EVAL_DATA, TARGET_LAYER, 1.0)
h_eval_dev  = h_eval_all.to(DEVICE)
labels      = l_eval_all.numpy()
eval_scores = probe.predict_proba(h_eval_all.numpy())[:, 1]

# identify eval boundary samples
eval_bm   = ((eval_scores >= BOUNDARY_LOW) &
              (eval_scores <= BOUNDARY_HIGH))
n_eval_b  = eval_bm.sum()
print(f"Eval boundary samples: {n_eval_b}/{len(eval_scores)} "
      f"({100*n_eval_b/len(eval_scores):.1f}%)")

# apply steering — boundary only
h_steered  = h_eval_dev.clone()
n_steered  = 0
policy.eval()
with torch.no_grad():
    for i in range(len(h_eval_dev)):
        ps = float(eval_scores[i])
        if BOUNDARY_LOW <= ps <= BOUNDARY_HIGH:
            h_steered[i] = policy.steer(
                h_eval_dev[i].unsqueeze(0)).squeeze(0)
            n_steered += 1

h_steered_cpu = h_steered.cpu()

# AUROC — all samples
from sklearn.model_selection import cross_val_score as cvs
a_before_all = cvs(
    LogisticRegression(max_iter=1000), h_eval_all.numpy(),
    labels, cv=5, scoring='roc_auc').mean()
a_after_all  = cvs(
    LogisticRegression(max_iter=1000), h_steered_cpu.numpy(),
    labels, cv=5, scoring='roc_auc').mean()

# AUROC — boundary samples only (main result)
h_eval_b    = h_eval_all[eval_bm]
h_steered_b = h_steered_cpu[eval_bm]
l_eval_b    = l_eval_all[eval_bm]

if n_eval_b > 10:
    a_before_b = cvs(
        LogisticRegression(max_iter=1000),
        h_eval_b.numpy(), l_eval_b.numpy(),
        cv=3, scoring='roc_auc').mean()
    a_after_b  = cvs(
        LogisticRegression(max_iter=1000),
        h_steered_b.numpy(), l_eval_b.numpy(),
        cv=3, scoring='roc_auc').mean()
else:
    a_before_b = a_after_b = 0.0

# projection gap — all samples
pd_cpu   = probe_dir.cpu()
pf_b_all = (h_eval_all[l_eval_all==1] @ pd_cpu).mean().item()
ph_b_all = (h_eval_all[l_eval_all==0] @ pd_cpu).mean().item()
pf_a_all = (h_steered_cpu[l_eval_all==1] @ pd_cpu).mean().item()
ph_a_all = (h_steered_cpu[l_eval_all==0] @ pd_cpu).mean().item()
gap_before = pf_b_all - ph_b_all
gap_after  = pf_a_all - ph_a_all

# projection gap — boundary only
if n_eval_b > 0:
    pf_b_b = (h_eval_b[l_eval_b==1] @ pd_cpu).mean().item() \
              if (l_eval_b==1).sum()>0 else 0
    ph_b_b = (h_eval_b[l_eval_b==0] @ pd_cpu).mean().item() \
              if (l_eval_b==0).sum()>0 else 0
    pf_a_b = (h_steered_b[l_eval_b==1] @ pd_cpu).mean().item() \
              if (l_eval_b==1).sum()>0 else 0
    ph_a_b = (h_steered_b[l_eval_b==0] @ pd_cpu).mean().item() \
              if (l_eval_b==0).sum()>0 else 0
    gap_before_b = pf_b_b - ph_b_b
    gap_after_b  = pf_a_b - ph_a_b
else:
    gap_before_b = gap_after_b = 0.0

# mean Δh magnitude on eval boundary samples
with torch.no_grad():
    deltas_eval = []
    for i in range(len(h_eval_dev)):
        ps = float(eval_scores[i])
        if BOUNDARY_LOW <= ps <= BOUNDARY_HIGH:
            d = policy(h_eval_dev[i].unsqueeze(0))
            deltas_eval.append(d.norm().item())
mean_delta_eval = np.mean(deltas_eval) if deltas_eval else 0

# early detection: raw vs steered
print("\nComputing early detection AUROC at each cutoff...")
early_boundary = {}
for cutoff in CUTOFFS:
    h_c, l_c    = extract_hidden_states(
        EVAL_DATA, TARGET_LAYER, cutoff)
    h_c_dev     = h_c.to(DEVICE)
    scores_c    = probe.predict_proba(h_c.numpy())[:, 1]
    boundary_c  = ((scores_c >= BOUNDARY_LOW) &
                   (scores_c <= BOUNDARY_HIGH))

    h_c_s = h_c_dev.clone()
    with torch.no_grad():
        for i in range(len(h_c_dev)):
            ps = float(scores_c[i])
            if BOUNDARY_LOW <= ps <= BOUNDARY_HIGH:
                h_c_s[i] = policy.steer(
                    h_c_dev[i].unsqueeze(0)).squeeze(0)

    raw_a, _   = robust_auroc(h_c,         l_c, cv=5)
    steer_a, _ = robust_auroc(h_c_s.cpu(), l_c, cv=5)

    # boundary-only AUROC at this cutoff
    if boundary_c.sum() > 5:
        h_c_b   = h_c[boundary_c]
        h_c_s_b = h_c_s.cpu()[boundary_c]
        l_c_b   = l_c[boundary_c]
        raw_b, _   = robust_auroc(h_c_b,   l_c_b, cv=3)
        steer_b, _ = robust_auroc(h_c_s_b, l_c_b, cv=3)
    else:
        raw_b = steer_b = 0.0

    early_boundary[cutoff] = {
        'raw_auroc':      raw_a,
        'steer_auroc':    steer_a,
        'delta':          steer_a - raw_a,
        'raw_boundary':   raw_b,
        'steer_boundary': steer_b,
        'delta_boundary': steer_b - raw_b,
        'n_boundary':     int(boundary_c.sum()),
    }

# save results
eval_results = {
    'auroc_before_all':      float(a_before_all),
    'auroc_after_all':       float(a_after_all),
    'auroc_before_boundary': float(a_before_b),
    'auroc_after_boundary':  float(a_after_b),
    'gap_before_all':        float(gap_before),
    'gap_after_all':         float(gap_after),
    'gap_before_boundary':   float(gap_before_b),
    'gap_after_boundary':    float(gap_after_b),
    'n_steered':             int(n_steered),
    'n_eval':                len(EVAL_DATA),
    'mean_delta':            float(mean_delta_eval),
    'early_boundary':        early_boundary,
}
with open(f'{SAVE_DIR}/eval_boundary_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

# print results
print(f"\n{'='*62}")
print(f"  BOUNDARY STEERING — EVALUATION RESULTS")
print(f"  Boundary zone: probe ∈ [{BOUNDARY_LOW}, {BOUNDARY_HIGH}]")
print(f"  L2 regularizer: λ = {LAMBDA_L2}")
print(f"{'='*62}")

print(f"\n── All eval samples ──────────────────────────────────")
print(f"  {'Metric':<35} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*62}")
print(f"  {'AUROC':<35} "
      f"{a_before_all:>8.4f} {a_after_all:>8.4f} "
      f"{a_after_all-a_before_all:>+8.4f}")
print(f"  {'Projection gap':<35} "
      f"{gap_before:>+8.4f} {gap_after:>+8.4f} "
      f"{gap_after-gap_before:>+8.4f}")

print(f"\n── Boundary samples only ({n_eval_b} samples) ────────")
print(f"  THIS IS THE MAIN RESULT — samples the policy targets")
print(f"  {'Metric':<35} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*62}")
print(f"  {'AUROC (boundary only)':<35} "
      f"{a_before_b:>8.4f} {a_after_b:>8.4f} "
      f"{a_after_b-a_before_b:>+8.4f}")
print(f"  {'Projection gap (boundary)':<35} "
      f"{gap_before_b:>+8.4f} {gap_after_b:>+8.4f} "
      f"{gap_after_b-gap_before_b:>+8.4f}")
print(f"\n  Mean |Δh| on boundary samples: {mean_delta_eval:.5f}")
print(f"  (small value = L2 regularizer working ✓)")

print(f"\n── Early detection ───────────────────────────────────")
print(f"  {'Cutoff':>8} {'Raw':>8} {'Steered':>10} "
      f"{'Δ All':>8} {'Δ Boundary':>12}")
print(f"  {'-'*52}")
for cutoff in CUTOFFS:
    r = early_boundary[cutoff]
    print(f"  {int(cutoff*100):>7}%  "
          f"{r['raw_auroc']:>8.4f}  "
          f"{r['steer_auroc']:>10.4f}  "
          f"{r['delta']:>+8.4f}  "
          f"{r['delta_boundary']:>+12.4f}")

print(f"\n  Samples steered: {n_steered}/{len(EVAL_DATA)} "
      f"({100*n_steered/len(EVAL_DATA):.1f}%)")

print(f"\n── Interpretation ────────────────────────────────────")
if a_after_b > a_before_b:
    print(f"  ✓ AUROC improved on boundary samples: "
          f"{a_before_b:.4f} → {a_after_b:.4f} "
          f"({a_after_b-a_before_b:+.4f})")
    print(f"    Ambiguous responses steered toward factual side")
elif gap_after_b > gap_before_b:
    print(f"  ~ AUROC unchanged but projection gap improved")
    print(f"    Boundary samples are more separated along probe direction")
else:
    print(f"  ~ No improvement on boundary samples")
    print(f"    Consider: wider boundary zone, more epochs, lower lambda")

if mean_delta_eval < 0.01:
    print(f"  ✓ L2 regularizer kept |Δh| small ({mean_delta_eval:.5f})")
    print(f"    Minimal perturbation — representation not corrupted")


### CELL 8 — Figures (5 plots) + final results table

Five publication-ready figures saved to Google Drive.

In [ ]:
# ── CELL 8 — Figures and final results ───────────────────────────
#
# 5 publication-ready figures saved to Google Drive.

# Figure 1: Layer AUROC curve
fig, ax = plt.subplots(figsize=(10, 4))
layers     = list(range(NUM_LAYERS + 1))
auroc_vals = [layer_aurocs[i] for i in layers]
ax.plot(layers, auroc_vals, 'o-', color='steelblue',
        linewidth=1.5, markersize=4)
ax.axvline(TARGET_LAYER, color='coral', linestyle='--',
           linewidth=1.5,
           label=f'L*={TARGET_LAYER} '
           f'({TARGET_LAYER/NUM_LAYERS*100:.0f}% depth)')
ax.scatter([TARGET_LAYER], [layer_aurocs[TARGET_LAYER]],
           color='coral', s=120, zorder=5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Layer', fontsize=11)
ax.set_ylabel('AUROC (probe)', fontsize=11)
ax.set_title(
    f'Figure 1: Critical layer discovery\n'
    f'L*={TARGET_LAYER} ({TARGET_LAYER/NUM_LAYERS*100:.0f}% depth) — '
    f'TriviaQA — Llama-3.2-3B',
    fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig1_layer_sweep.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved ✓")

# Figure 2: Early detection
fig, ax = plt.subplots(figsize=(8, 4))
xi     = range(len(CUTOFFS))
aurocs = [early_results[c]['auroc'] for c in CUTOFFS]
stds   = [early_results[c]['std']   for c in CUTOFFS]
ax.plot(xi, aurocs, 'o-', color='steelblue',
        linewidth=2, markersize=8)
ax.fill_between(xi,
    [a-s for a, s in zip(aurocs, stds)],
    [a+s for a, s in zip(aurocs, stds)],
    alpha=0.15, color='steelblue')
ax.axhline(0.5, color='gray', linestyle='--',
           alpha=0.5, label='Chance')
ax.axhline(0.65, color='green', linestyle='--',
           alpha=0.5, label='0.65 threshold')
ax.set_xticks(xi)
ax.set_xticklabels([f"{int(c*100)}%" for c in CUTOFFS],
                   fontsize=11)
ax.set_xlabel('Tokens generated', fontsize=11)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_title(
    f'Figure 2: Early detection at L*={TARGET_LAYER}\n'
    f'How early can hallucination be detected?',
    fontsize=11)
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig2_early_detection.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✓")

# Figure 3: PCA before vs after steering
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
y_plot    = labels
for ax, h_plot, title in [
    (axes[0], h_eval_all,     f'Before steering (L*={TARGET_LAYER})'),
    (axes[1], h_steered_cpu,  'After boundary RL steering')
]:
    reduced = PCA(n_components=2).fit_transform(h_plot.numpy())
    fm_p    = y_plot == 1
    # non-boundary in lighter color
    ax.scatter(reduced[~eval_bm & fm_p,  0],
               reduced[~eval_bm & fm_p,  1],
               c='lightblue', alpha=0.3, s=12,
               label='Factual (confident)')
    ax.scatter(reduced[~eval_bm & ~fm_p, 0],
               reduced[~eval_bm & ~fm_p, 1],
               c='lightsalmon', alpha=0.3, s=12,
               label='Hallu (confident)')
    # boundary in full color
    ax.scatter(reduced[eval_bm & fm_p,  0],
               reduced[eval_bm & fm_p,  1],
               c='steelblue', alpha=0.9, s=40,
               label='Factual (boundary)')
    ax.scatter(reduced[eval_bm & ~fm_p, 0],
               reduced[eval_bm & ~fm_p, 1],
               c='coral', alpha=0.9, s=40,
               label='Hallu (boundary)')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, loc='best')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle(
    'Figure 3: Boundary samples highlighted — before vs after steering\n'
    f'Boundary zone: probe ∈ [{BOUNDARY_LOW}, {BOUNDARY_HIGH}]',
    fontsize=11)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig3_pca_boundary.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved ✓")

# Figure 4: Training log
if train_log:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ep         = [t['epoch']        for t in train_log]
    gaps       = [t['boundary_gap'] for t in train_log]
    aurocs_log = [t['auroc']        for t in train_log]
    rews       = [t['steer_rew']    for t in train_log]
    deltas_log = [t['mean_delta']   for t in train_log]

    axes[0].plot(ep, gaps, 'o-', color='coral', linewidth=1.5)
    axes[0].axhline(init_gap, color='gray', linestyle='--',
                    alpha=0.5, label=f'Init ({init_gap:.4f})')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Boundary gap')
    axes[0].set_title('Figure 4a: Boundary projection gap\n'
                      '(increasing = boundary samples moving to factual)')
    axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

    axes[1].plot(ep, rews, 'o-', color='steelblue', linewidth=1.5)
    axes[1].axhline(0, color='gray', linestyle=':', alpha=0.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Steering reward')
    axes[1].set_title('Figure 4b: Steering reward\n'
                      '(positive = moving in correct direction)')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(ep, deltas_log, 'o-', color='purple', linewidth=1.5)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean |Δh|')
    axes[2].set_title(f'Figure 4c: Perturbation magnitude\n'
                      f'(L2 λ={LAMBDA_L2} keeps this small)')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/fig4_training.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Figure 4 saved ✓")

# Figure 5: Early detection comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xi       = range(len(CUTOFFS))
x_labels = [f"{int(c*100)}%" for c in CUTOFFS]

# left: all samples
raw_all   = [early_boundary[c]['raw_auroc']   for c in CUTOFFS]
steer_all = [early_boundary[c]['steer_auroc'] for c in CUTOFFS]
axes[0].plot(xi, raw_all,   'o-', color='steelblue',
             linewidth=2, markersize=7, label='Raw')
axes[0].plot(xi, steer_all, 's-', color='coral',
             linewidth=2, markersize=7, label='After boundary steering')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xticks(xi); axes[0].set_xticklabels(x_labels)
axes[0].set_ylabel('AUROC'); axes[0].set_ylim(0.4, 1.0)
axes[0].set_title('Figure 5a: All samples')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# right: boundary samples only
raw_b     = [early_boundary[c]['raw_boundary']   for c in CUTOFFS]
steer_b   = [early_boundary[c]['steer_boundary'] for c in CUTOFFS]
axes[1].plot(xi, raw_b,   'o-', color='steelblue',
             linewidth=2, markersize=7, label='Raw (boundary)')
axes[1].plot(xi, steer_b, 's-', color='coral',
             linewidth=2, markersize=7,
             label='After steering (boundary)')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xticks(xi); axes[1].set_xticklabels(x_labels)
axes[1].set_ylabel('AUROC'); axes[1].set_ylim(0.4, 1.0)
axes[1].set_title('Figure 5b: Boundary samples only\n'
                  '(main result — samples the policy targets)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Figure 5: Early detection — raw vs RL boundary steering',
             fontsize=12)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig5_early_detection_boundary.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved ✓")

# Final results table
print(f"\n{'='*62}")
print(f"  FINAL RESULTS — TriviaQA + Boundary RL Steering")
print(f"{'='*62}")
print(f"\nModel    : Llama-3.2-3B-Instruct (frozen)")
print(f"Layer L* : {TARGET_LAYER} ({TARGET_LAYER/NUM_LAYERS*100:.0f}% depth)")
print(f"Method   : Boundary-only RL steering + probe direction + L2 reg")

print(f"\n── Table 1: Baseline probe (Cell 3-4) ───────────────")
print(f"  {'Cutoff':>8} {'AUROC':>8} {'±Std':>8}")
print(f"  {'-'*28}")
for cutoff in CUTOFFS:
    r = early_results[cutoff]
    print(f"  {int(cutoff*100):>7}%  {r['auroc']:>8.4f}  "
          f"±{r['std']:.4f}")

print(f"\n── Table 2: RL boundary steering (Cell 6-7) ─────────")
print(f"  Boundary zone: [{BOUNDARY_LOW}, {BOUNDARY_HIGH}]  "
      f"L2 λ={LAMBDA_L2}")
print(f"\n  All samples:")
print(f"  {'Metric':<30} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*56}")
print(f"  {'AUROC':<30} "
      f"{a_before_all:>8.4f} {a_after_all:>8.4f} "
      f"{a_after_all-a_before_all:>+8.4f}")
print(f"  {'Projection gap':<30} "
      f"{gap_before:>+8.4f} {gap_after:>+8.4f} "
      f"{gap_after-gap_before:>+8.4f}")

print(f"\n  Boundary samples only ({n_eval_b} samples):")
print(f"  {'Metric':<30} {'Before':>8} {'After':>8} {'Delta':>8}")
print(f"  {'-'*56}")
print(f"  {'AUROC':<30} "
      f"{a_before_b:>8.4f} {a_after_b:>8.4f} "
      f"{a_after_b-a_before_b:>+8.4f}")
print(f"  {'Projection gap':<30} "
      f"{gap_before_b:>+8.4f} {gap_after_b:>+8.4f} "
      f"{gap_after_b-gap_before_b:>+8.4f}")
print(f"\n  Mean |Δh| (L2 regularizer): {mean_delta_eval:.5f}")
print(f"  Samples steered: {n_steered}/{len(EVAL_DATA)} "
      f"({100*n_steered/len(EVAL_DATA):.1f}%)")

print(f"\n── Files saved ───────────────────────────────────────")
for fname in [
    'labeled_data.json', 'layer_sweep.json',
    'early_detection.json', 'training_log_boundary.json',
    'eval_boundary_results.json', 'steering_boundary_best.pt',
    'fig1_layer_sweep.png', 'fig2_early_detection.png',
    'fig3_pca_boundary.png', 'fig4_training.png',
    'fig5_early_detection_boundary.png',
]:
    path = f'{SAVE_DIR}/{fname}'
    if os.path.exists(path):
        print(f"  {fname:<45} ✓")

print(f"\n{'='*62}")
print(f"  DONE")
print(f"{'='*62}")


verification and fixing


In [ ]:
# check probe score distribution on eval set
h_eval_check, l_eval_check = extract_hidden_states(
    EVAL_DATA, TARGET_LAYER, 1.0)
scores_check = probe.predict_proba(
    h_eval_check.numpy())[:, 1]

print("Probe score distribution on eval set:")
bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.45,
        0.55, 0.65, 0.7, 0.8, 0.9, 1.0]
for i in range(len(bins)-1):
    lo, hi  = bins[i], bins[i+1]
    count   = ((scores_check >= lo) & (scores_check < hi)).sum()
    bar     = '█' * int(count / 2)
    zone    = " ← boundary" if 0.45 <= lo < 0.65 else ""
    print(f"  [{lo:.2f}-{hi:.2f}): {count:>4}  {bar}{zone}")

print(f"\nTotal eval: {len(scores_check)}")
for low, high in [(0.35, 0.75), (0.40, 0.70),
                  (0.45, 0.65), (0.30, 0.80)]:
    n = ((scores_check >= low) &
         (scores_check <= high)).sum()
    f_count = ((scores_check >= low) &
               (scores_check <= high) &
               (l_eval_check.numpy() == 1)).sum()
    h_count = ((scores_check >= low) &
               (scores_check <= high) &
               (l_eval_check.numpy() == 0)).sum()
    print(f"  Zone [{low}-{high}]: {n} samples "
          f"(factual={f_count}, hallu={h_count})")

In [ ]:
for low, high in [(0.45,0.65), (0.40,0.70), (0.35,0.75), (0.30,0.80)]:
    mask = (scores_check >= low) & (scores_check <= high)
    y = l_eval_check[mask]
    s = scores_check[mask]
    print(f"\nZone [{low},{high}]")
    print(" count:", len(y))
    print(" factual mean score:", s[y == 1].mean() if (y == 1).sum() else None)
    print(" hallu   mean score:", s[y == 0].mean() if (y == 0).sum() else None)

In [ ]:
from sklearn.metrics import roc_auc_score

for low, high in [(0.45,0.65), (0.40,0.70), (0.35,0.75), (0.30,0.80)]:
    mask = (scores_check >= low) & (scores_check <= high)
    y = l_eval_check[mask]
    s = scores_check[mask]
    if len(set(y.tolist())) == 2:
        print(low, high, roc_auc_score(y, s))
    else:
        print(low, high, "single-class subset")

In [ ]:
from sklearn.metrics import roc_auc_score

zones = [(0.45,0.65), (0.40,0.70), (0.35,0.75), (0.30,0.80)]

for low, high in zones:
    mask = (scores_check >= low) & (scores_check <= high)
    y = l_eval_check[mask]
    s = scores_check[mask]

    print(f"\nZone [{low},{high}]")
    print("n =", len(y), " factual =", (y==1).sum(), " hallu =", (y==0).sum())
    print("score min/max =", float(s.min()), float(s.max()))
    print("auc(score) =", roc_auc_score(y, s))
    print("auc(1-score) =", roc_auc_score(y, 1 - s))